In [1]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, utils, models
from torch.utils.data import DataLoader
import numpy as np

# =========================
# 2. USER INPUT PARAMETERS
# =========================
dataset_choice = input("Enter dataset (mnist / fashion): ").lower()
epochs = int(input("Enter number of epochs: "))
batch_size = int(input("Enter batch size: "))
noise_dim = int(input("Enter noise dimension: "))
learning_rate = float(input("Enter learning rate: "))
save_interval = int(input("Enter save interval: "))

# =========================
# 3. DEVICE CONFIGURATION
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 4. DATASET LOADING
# =========================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])   # scale to [-1, 1]
])

if dataset_choice == "mnist":
    dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
elif dataset_choice == "fashion":
    dataset = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
else:
    raise ValueError("Invalid dataset choice")

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
img_shape = (1, 28, 28)

# =========================
# 5. GENERATOR MODEL
# =========================
class Generator(nn.Module):
    def __init__(self, noise_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, 28 * 28),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        return img.view(z.size(0), *img_shape)

# =========================
# 6. DISCRIMINATOR MODEL
# =========================
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.model(img)

# =========================
# 7. MODEL INITIALIZATION
# =========================
generator = Generator(noise_dim).to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

# =========================
# 8. OUTPUT DIRECTORIES
# =========================
os.makedirs("generated_samples", exist_ok=True)
os.makedirs("final_generated_images", exist_ok=True)

# =========================
# 9. TRAINING LOOP
# =========================
for epoch in range(1, epochs + 1):
    d_loss_epoch = 0
    g_loss_epoch = 0
    correct = 0
    total = 0

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        bs = real_imgs.size(0)

        real_labels = torch.ones(bs, 1).to(device)
        fake_labels = torch.zeros(bs, 1).to(device)

        # ---- Train Discriminator ----
        optimizer_D.zero_grad()

        real_preds = discriminator(real_imgs)
        d_real_loss = criterion(real_preds, real_labels)

        noise = torch.randn(bs, noise_dim).to(device)
        fake_imgs = generator(noise)
        fake_preds = discriminator(fake_imgs.detach())
        d_fake_loss = criterion(fake_preds, fake_labels)

        d_loss = d_real_loss + d_fake_loss
        d_loss.backward()
        optimizer_D.step()

        preds = torch.cat([real_preds, fake_preds])
        labels = torch.cat([real_labels, fake_labels])
        correct += (preds.round() == labels).sum().item()
        total += labels.size(0)

        # ---- Train Generator ----
        optimizer_G.zero_grad()
        fake_preds = discriminator(fake_imgs)
        g_loss = criterion(fake_preds, real_labels)
        g_loss.backward()
        optimizer_G.step()

        d_loss_epoch += d_loss.item()
        g_loss_epoch += g_loss.item()

    d_acc = 100 * correct / total

    print(f"Epoch {epoch}/{epochs} | D_loss: {d_loss_epoch:.2f} | D_acc: {d_acc:.2f}% | G_loss: {g_loss_epoch:.2f}")

    # ---- Save generated samples ----
    if epoch % save_interval == 0:
        utils.save_image(fake_imgs[:25],
                         f"generated_samples/epoch_{epoch:02d}.png",
                         nrow=5,
                         normalize=True)

# =========================
# 10. FINAL IMAGE GENERATION
# =========================
generator.eval()
noise = torch.randn(100, noise_dim).to(device)
final_images = generator(noise)

for i in range(100):
    utils.save_image(final_images[i],
                     f"final_generated_images/img_{i+1}.png",
                     normalize=True)

# =========================
# 11. LABEL PREDICTION (TRANSFER LEARNING)
# =========================
classifier = models.resnet18(pretrained=True)
classifier.fc = nn.Linear(classifier.fc.in_features, 10)
classifier = classifier.to(device)
classifier.eval()

predicted_labels = []

with torch.no_grad():
    for img in final_images:
        img = img.repeat(3, 1, 1)  # 1-channel → 3-channel
        img = img.unsqueeze(0).to(device)
        output = classifier(img)
        predicted_labels.append(torch.argmax(output, dim=1).item())

# =========================
# 12. LABEL DISTRIBUTION
# =========================
print("\nLabel Distribution of Generated Images:")
unique, counts = np.unique(predicted_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Label {u}: {c}")

Enter dataset (mnist / fashion): mnist
Enter number of epochs: 10
Enter batch size: 20
Enter noise dimension: 100
Enter learning rate: 0.0002
Enter save interval: 5
Using device: cuda


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.65MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 133kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.80MB/s]


Epoch 1/10 | D_loss: 2347.12 | D_acc: 83.00% | G_loss: 7418.27
Epoch 2/10 | D_loss: 1650.17 | D_acc: 91.34% | G_loss: 10468.01
Epoch 3/10 | D_loss: 1199.26 | D_acc: 93.99% | G_loss: 12132.60
Epoch 4/10 | D_loss: 332.38 | D_acc: 98.53% | G_loss: 18799.58
Epoch 5/10 | D_loss: 96.61 | D_acc: 99.87% | G_loss: 34087.19
Epoch 6/10 | D_loss: 44.29 | D_acc: 99.82% | G_loss: 80252.54
Epoch 7/10 | D_loss: 14.61 | D_acc: 99.95% | G_loss: 260789.96
Epoch 8/10 | D_loss: 0.00 | D_acc: 100.00% | G_loss: 300000.00
Epoch 9/10 | D_loss: 0.00 | D_acc: 100.00% | G_loss: 300000.00
Epoch 10/10 | D_loss: 0.00 | D_acc: 100.00% | G_loss: 300000.00


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 218MB/s]



Label Distribution of Generated Images:
Label 9: 100
